# Notebook 02 (Participant): Evaluation + Metrics

You will implement the evaluation layer and produce the evidence used for scientific comparison.


**Edit-safe start:** this notebook opens from GitHub in read-only source mode. Use **File -> Save a copy in Drive** before running edits so your changes stay in your own workspace.


## Notebook map

This notebook is written as a standalone lab chapter:
- context first,
- implementation second,
- interpretation third.

If you are following asynchronously, run cells in order and use the success checks to validate each stage before moving on.

### Public exercise legend
- `PUBLIC FILL-IN CELL`: complete this block directly.
- `CHECKPOINT`: verify before moving forward.
- Metric interpretation is as important as metric computation.


## Standalone guide

This chapter answers: *did the generated designs actually improve engineering outcomes under benchmark simulation?*


In [ ]:
# Colab/local dependency bootstrap
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
FORCE_INSTALL = False  # Set True to force install outside Colab


def pip_install(packages: list[str]):
    cmd = [sys.executable, "-m", "pip", "install", *packages]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)


BASE_PACKAGES = ["engibench[beams2d]", "sqlitedict", "matplotlib", "tqdm", "tyro", "wandb"]
ENGIOPT_GIT = "git+https://github.com/IDEALLab/EngiOpt.git@codex/dcc26-workshop-notebooks#egg=engiopt"

if IN_COLAB or FORCE_INSTALL:
    print("Installing dependencies...")
    pip_install(BASE_PACKAGES)
    pip_install([ENGIOPT_GIT])

    try:
        import torch  # noqa: F401
    except Exception:
        pip_install(["torch", "torchvision"])

    print("Dependency install complete.")
else:
    print("Skipping install (using current environment). Set FORCE_INSTALL=True to install here.")

## Artifact loading

Load artifacts from Notebook 01, optional W&B, or local auto-build fallback.
Do not proceed until artifact shapes/configs are confirmed.


### Why these metrics matter for benchmarking

Objective alone can overstate progress.
We include feasibility, diversity, and novelty proxies to reduce that blind spot.


### Step 1 - Resolve artifact source and recovery path

Checkpoint: you can load `generated`, `baseline`, and `conditions` with matching sample counts.


In [ ]:
import json
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch as th
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from engibench.problems.beams2d.v0 import Beams2D

try:
    from engiopt.cgan_2d.cgan_2d import Generator as EngiOptCGAN2DGenerator
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Could not import engiopt model class. Run the bootstrap cell first; on Colab, restart runtime after install if needed."
    ) from exc

USE_WANDB_ARTIFACTS = False
WANDB_PROJECT = "dcc26-workshop"
WANDB_ENTITY = None
WANDB_ARTIFACT_NAME = "dcc26_beams2d_generated_artifacts"
WANDB_ARTIFACT_ALIAS = "latest"

# Self-heal path for workshop robustness
AUTO_BUILD_ARTIFACTS_IF_MISSING = True


def resolve_artifact_dir(create: bool = False) -> Path:
    in_colab = "google.colab" in sys.modules
    path = Path("/content/dcc26_artifacts") if in_colab else Path("workshops/dcc26/artifacts")
    if create:
        path.mkdir(parents=True, exist_ok=True)
    return path


def build_artifacts_locally(
    artifact_dir: Path,
    seed: int = 7,
    n_train: int = 512,
    n_samples: int = 24,
    epochs: int = 8,
    batch_size: int = 64,
    latent_dim: int = 32,
) -> None:
    print("Building Notebook 01-style artifacts locally with EngiOpt...")

    random.seed(seed)
    np.random.seed(seed)
    th.manual_seed(seed)
    if th.cuda.is_available():
        th.cuda.manual_seed_all(seed)

    device = th.device("cuda" if th.cuda.is_available() else "cpu")
    problem = Beams2D(seed=seed)
    train_ds = problem.dataset["train"]
    test_ds = problem.dataset["test"]
    condition_keys = problem.conditions_keys

    rng = np.random.default_rng(seed)
    subset_size = min(n_train, len(train_ds))
    subset_idx = rng.choice(len(train_ds), size=subset_size, replace=False)

    conds_np = np.stack([np.array(train_ds[k])[subset_idx].astype(np.float32) for k in condition_keys], axis=1)
    designs_np = np.array(train_ds["optimal_design"])[subset_idx].astype(np.float32)
    targets_np = designs_np * 2.0 - 1.0

    model = EngiOptCGAN2DGenerator(
        latent_dim=latent_dim,
        n_conds=conds_np.shape[1],
        design_shape=problem.design_space.shape,
    ).to(device)
    optimizer = th.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()

    def sample_noise(batch: int) -> th.Tensor:
        return th.randn((batch, latent_dim), device=device, dtype=th.float32)

    ds = TensorDataset(th.tensor(conds_np), th.tensor(targets_np))
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True)

    train_losses = []
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for cond_batch, target_batch in dl:
            cond_batch = cond_batch.to(device)
            target_batch = target_batch.to(device)

            pred = model(sample_noise(cond_batch.shape[0]), cond_batch)
            loss = criterion(pred, target_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += float(loss.item())

        epoch_avg = epoch_loss / len(dl)
        train_losses.append(epoch_avg)
        print(f"bootstrap epoch {epoch + 1:02d}/{epochs} - loss: {epoch_avg:.4f}")

    sample_count = min(n_samples, len(test_ds))
    selected = rng.choice(len(test_ds), size=sample_count, replace=False)
    test_conds = np.stack([np.array(test_ds[k])[selected].astype(np.float32) for k in condition_keys], axis=1)
    baseline_designs = np.array(test_ds["optimal_design"])[selected].astype(np.float32)

    model.eval()
    with th.no_grad():
        tanh_out = model(sample_noise(sample_count), th.tensor(test_conds, device=device))
        gen_designs_t = ((tanh_out.clamp(-1.0, 1.0) + 1.0) / 2.0).clamp(0.0, 1.0)
    gen_designs = gen_designs_t.detach().cpu().numpy().astype(np.float32)

    conditions_records = []
    for i in range(sample_count):
        rec = {}
        for j, k in enumerate(condition_keys):
            v = test_conds[i, j]
            rec[k] = bool(v) if k == "overhang_constraint" else float(v)
        conditions_records.append(rec)

    artifact_dir.mkdir(parents=True, exist_ok=True)
    np.save(artifact_dir / "generated_designs.npy", gen_designs)
    np.save(artifact_dir / "baseline_designs.npy", baseline_designs)
    with open(artifact_dir / "conditions.json", "w", encoding="utf-8") as f:
        json.dump(conditions_records, f, indent=2)

    pd.DataFrame({"epoch": np.arange(1, len(train_losses) + 1), "train_loss": train_losses}).to_csv(
        artifact_dir / "training_history.csv", index=False
    )

    th.save(
        {
            "model": model.state_dict(),
            "condition_keys": condition_keys,
            "latent_dim": latent_dim,
            "model_family": "engiopt.cgan_2d.Generator",
        },
        artifact_dir / "engiopt_cgan2d_generator_supervised.pt",
    )

    print("Built artifacts at", artifact_dir)


ARTIFACT_DIR = resolve_artifact_dir(create=True)
required = [
    ARTIFACT_DIR / "generated_designs.npy",
    ARTIFACT_DIR / "baseline_designs.npy",
    ARTIFACT_DIR / "conditions.json",
]

if not all(p.exists() for p in required):
    if USE_WANDB_ARTIFACTS:
        try:
            import wandb

            run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, job_type="artifact-download", reinit=True)
            if WANDB_ENTITY:
                artifact_ref = f"{WANDB_ENTITY}/{WANDB_PROJECT}/{WANDB_ARTIFACT_NAME}:{WANDB_ARTIFACT_ALIAS}"
            else:
                artifact_ref = f"{WANDB_PROJECT}/{WANDB_ARTIFACT_NAME}:{WANDB_ARTIFACT_ALIAS}"
            artifact = run.use_artifact(artifact_ref, type="dataset")
            artifact.download(root=str(ARTIFACT_DIR))
            run.finish()
            print("Downloaded artifacts from W&B to", ARTIFACT_DIR)
        except Exception as exc:
            if AUTO_BUILD_ARTIFACTS_IF_MISSING:
                print("W&B download failed; switching to local artifact build:", exc)
                build_artifacts_locally(ARTIFACT_DIR)
            else:
                raise FileNotFoundError(
                    "Artifacts missing locally and W&B download failed. "
                    "Run Notebook 01 first or disable USE_WANDB_ARTIFACTS. "
                    f"Details: {exc}"
                ) from exc
    elif AUTO_BUILD_ARTIFACTS_IF_MISSING:
        build_artifacts_locally(ARTIFACT_DIR)
    else:
        missing = "\n".join(f"- {p}" for p in required if not p.exists())
        raise FileNotFoundError(
            "Notebook 01 artifacts not found. Run Notebook 01 first (including export cell), "
            "or enable USE_WANDB_ARTIFACTS to fetch from W&B. Missing files:\\n" + missing
        )

print("using artifact dir:", ARTIFACT_DIR)

gen_designs = np.load(ARTIFACT_DIR / "generated_designs.npy")
baseline_designs = np.load(ARTIFACT_DIR / "baseline_designs.npy")
with open(ARTIFACT_DIR / "conditions.json", encoding="utf-8") as f:
    conditions = json.load(f)

print("generated:", gen_designs.shape)
print("baseline:", baseline_designs.shape)
print("conditions:", len(conditions))

### Step 2 - Implement per-sample evaluation (PUBLIC FILL-IN)

Compute constraint violations + objective values for generated and baseline designs under identical conditions.

Success criteria:
- one row per sample,
- objective values and violation counts both captured.


In [ ]:
problem = Beams2D(seed=7)

# PUBLIC FILL-IN CELL 02-A
# Goal: evaluate each sample pair with identical condition config.

rows = []

# START FILL ---------------------------------------------------------------
# for i, (g, b, cfg_raw) in enumerate(zip(gen_designs, baseline_designs, conditions, strict=True)):
#     cfg = dict(cfg_raw)
#     g_viol = problem.check_constraints(design=g, config=cfg)
#     b_viol = problem.check_constraints(design=b, config=cfg)
#
#     # reset between simulations for reproducibility / simulator hygiene
#     problem.reset(seed=7 + i)
#     g_obj = float(problem.simulate(design=g, config=cfg))
#     problem.reset(seed=7 + i)
#     b_obj = float(problem.simulate(design=b, config=cfg))
#
#     rows.append({
#         'sample': i,
#         'gen_obj': g_obj,
#         'base_obj': b_obj,
#         'gen_minus_base': g_obj - b_obj,
#         'gen_violations': len(g_viol),
#         'base_violations': len(b_viol),
#     })
raise NotImplementedError("Implement per-sample evaluation loop")
# END FILL -----------------------------------------------------------------

results = pd.DataFrame(rows)
results.head()

# CHECKPOINT
expected_cols = {"sample", "gen_obj", "base_obj", "gen_minus_base", "gen_violations", "base_violations"}
missing_cols = expected_cols.difference(results.columns)
assert not missing_cols, f"Missing result columns: {missing_cols}"
assert len(results) == len(gen_designs), "results must have one row per sample"
print("Checkpoint passed: per-sample evaluation table is complete.")

### Step 3 - Implement summary metrics (PUBLIC FILL-IN)

Report objective, feasibility, diversity, and novelty in a single table.

Success criteria:
- `summary_df` has one row,
- each metric has a clear interpretation for workshop discussion.


In [ ]:
# PUBLIC FILL-IN CELL 02-B
# Goal: aggregate per-sample metrics into one benchmark summary row.


def mean_pairwise_l2(designs: np.ndarray) -> float:
    flat = designs.reshape(designs.shape[0], -1)
    n = flat.shape[0]
    if n < 2:
        return 0.0
    dists = []
    for i in range(n):
        for j in range(i + 1, n):
            dists.append(float(np.linalg.norm(flat[i] - flat[j])))
    return float(np.mean(dists))


def mean_nn_distance_to_reference(designs: np.ndarray, reference_designs: np.ndarray) -> float:
    q = designs.reshape(designs.shape[0], -1)
    r = reference_designs.reshape(reference_designs.shape[0], -1)
    nn_dists = []
    for i in range(q.shape[0]):
        d = np.linalg.norm(r - q[i][None, :], axis=1)
        nn_dists.append(float(np.min(d)))
    return float(np.mean(nn_dists))


# START FILL ---------------------------------------------------------------
# Build one summary dict with at least these keys:
# - n_samples
# - gen_obj_mean
# - base_obj_mean
# - objective_gap_mean
# - improvement_rate
# - gen_violation_ratio
# - base_violation_ratio
# - gen_feasible_rate
# - gen_diversity_l2
# - gen_novelty_to_train_l2
raise NotImplementedError("Implement summary metric dictionary + summary_df")
# END FILL -----------------------------------------------------------------

# CHECKPOINT
assert "summary_df" in locals(), "Define summary_df"
assert len(summary_df) == 1, "summary_df should be one-row summary"
required_summary = {
    "n_samples",
    "gen_obj_mean",
    "base_obj_mean",
    "objective_gap_mean",
    "improvement_rate",
    "gen_violation_ratio",
    "base_violation_ratio",
    "gen_feasible_rate",
    "gen_diversity_l2",
    "gen_novelty_to_train_l2",
}
missing = required_summary.difference(summary_df.columns)
assert not missing, f"Missing summary columns: {missing}"
print("Checkpoint passed: summary table is ready for export/discussion.")

### Step 4 - Export evidence artifacts

Persist outputs as audit-ready artifacts, not ephemeral notebook prints.


In [ ]:
# Export metrics and figures
results_path = ARTIFACT_DIR / "per_sample_metrics.csv"
summary_path = ARTIFACT_DIR / "metrics_summary.csv"
hist_path = ARTIFACT_DIR / "objective_histogram.png"
grid_path = ARTIFACT_DIR / "design_grid.png"
scatter_path = ARTIFACT_DIR / "objective_scatter.png"

results.to_csv(results_path, index=False)
summary_df.to_csv(summary_path, index=False)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(results["gen_obj"], bins=10, alpha=0.7, label="generated")
ax.hist(results["base_obj"], bins=10, alpha=0.7, label="baseline")
ax.set_xlabel("Compliance objective (lower is better)")
ax.set_ylabel("Count")
ax.set_title("Generated vs baseline objective distribution")
ax.legend()
fig.tight_layout()
fig.savefig(hist_path, dpi=150)
plt.show()

fig2, ax2 = plt.subplots(figsize=(5, 5))
ax2.scatter(results["base_obj"], results["gen_obj"], alpha=0.8)
min_v = min(results["base_obj"].min(), results["gen_obj"].min())
max_v = max(results["base_obj"].max(), results["gen_obj"].max())
ax2.plot([min_v, max_v], [min_v, max_v], "--", color="black", linewidth=1)
ax2.set_xlabel("Baseline objective")
ax2.set_ylabel("Generated objective")
ax2.set_title("Per-sample objective comparison")
fig2.tight_layout()
fig2.savefig(scatter_path, dpi=150)
plt.show()

print("Saved:")
print("-", results_path)
print("-", summary_path)
print("-", hist_path)
print("-", scatter_path)

# Optional advanced extension:
# if USE_WANDB_ARTIFACTS:
#     log summary metrics, tables, and images to W&B.

### Step 5 - Visual comparison for interpretation

Use visuals to interpret outliers and reconcile metric-level contradictions.


In [ ]:
# Visual side-by-side sample grid
fig, axes = plt.subplots(3, 4, figsize=(12, 8))
for i, ax in enumerate(axes.ravel()):
    if i >= 12:
        break
    pair_idx = i // 2
    if i % 2 == 0:
        ax.imshow(gen_designs[pair_idx], cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"gen {pair_idx}")
    else:
        ax.imshow(baseline_designs[pair_idx], cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"base {pair_idx}")
    ax.axis("off")
fig.tight_layout()
fig.savefig(grid_path, dpi=150)
plt.show()

## Interpretation hints

Strong results usually balance objective quality **and** feasibility.
Use diversity/novelty to discuss exploration vs imitation behavior.


## Discussion bridge to workshop breakout

Bring one concrete claim and one uncertainty to discussion.
Example: “Objective improved, but feasibility degraded under stricter conditions.”


## Troubleshooting

If a section fails, do not continue downstream. Fix locally first, then rerun the section and its immediate checks.
This notebook is intentionally staged so failures are localized.


## Takeaways

Before closing, record three points:
1. What conclusion is directly supported by your metrics?
2. What remains uncertain (and why)?
3. What extra experiment would you run next to reduce that uncertainty?
